# Analysis of Music Listening in Moscow and St. Petersburg

Working hypotheses:

* people in the two cities, Moscow and St. Petersburg, listen to music on different schedules

* the top-10 genre lists for Monday morning and Friday evening are noticeably different

* people in the two cities prefer different music genres

# Getting the data

In [1]:
import pandas as pd

The file *music_of_big_cities.csv* is read into the variable *df*.

In [2]:
df = pd.read_csv('music_of_big_cities.csv')

In [3]:
df.head(10)

,Unnamed: 0,userID,Track,artist,genre,City,time,Day
0,0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Saint-Petersburg,20:28:33,Wednesday
1,1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Moscow,14:07:09,Friday
2,2,20EC38,Funiculì funiculà,Mario Lanza,pop,Saint-Petersburg,20:58:07,Wednesday
3,3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Saint-Petersburg,08:37:09,Monday
4,4,E2DC1FAE,Soul People,Space Echo,dance,Moscow,08:34:34,Monday
5,5,842029A1,Преданная,IMPERVTOR,rusrap,Saint-Petersburg,13:09:41,Friday
6,6,4CB90AA5,True,Roman Messer,dance,Moscow,13:00:07,Wednesday
7,7,F03E1C1F,Feeling This Way,Polina Griffith,dance,Moscow,20:47:49,Wednesday
8,8,8FA1D3BE,И вновь продолжается бой,NaN,ruspop,Moscow,09:17:40,Friday
9,9,E772D5C0,Pessimist,NaN,dance,Saint-Petersburg,21:20:49,Wednesday


General information about the data in df.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 8 columns):
Unnamed: 0    65079 non-null int64
  userID      65079 non-null object
Track         63848 non-null object
artist        57876 non-null object
genre         63881 non-null object
  City        65079 non-null object
time          65079 non-null object
Day           65079 non-null object
dtypes: int64(1), object(7)
memory usage: 4.0+ MB


A closer look at this information.

The table has 7 columns:

* userID: user ID;
* Track: track title;
* artist: artist name;
* genre: genre name;
* City: the city where the listening took place;
* time: the time when the user listened to the track;
* Day: day of the week.

Each row describes a track of a certain genre by a certain artist that users listened to in one of the cities at a certain time and day of the week. The *time*, *day*, and *City* columns are especially valuable for testing the working hypotheses. The *genre* column will show the most popular genres.

# Data preprocessing

Next: handling missing values, renaming the columns, and checking the data for duplicates.

In [5]:
df.columns

Index(['Unnamed: 0', '  userID', 'Track', 'artist', 'genre', '  City  ',
       'time', 'Day'],
      dtype='object')

The column names contain spaces, which can make the data harder to access. The columns are renamed to make further work easier, and the result is checked.

In [6]:
df = df.drop(df.columns[0], axis='columns')

In [7]:
df.columns = ['user_id','track_name','artist_name','genre_name','city','time','weekday']
df.head()

,user_id,track_name,artist_name,genre_name,city,time,weekday
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Saint-Petersburg,20:28:33,Wednesday
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Moscow,14:07:09,Friday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Saint-Petersburg,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Saint-Petersburg,08:37:09,Monday
4,E2DC1FAE,Soul People,Space Echo,dance,Moscow,08:34:34,Monday


Missing values are counted by chaining methods that add them up.

In [8]:
# count the total number of missing values in df
df.isnull().sum()

user_id           0
track_name     1231
artist_name    7203
genre_name     1198
city              0
time              0
weekday           0
dtype: int64

Missing values mean that not all information is available for some tracks. There can be different reasons: for example, a folk song has no specific artist, or there were problems recording the data. Each case needs to be examined to find the cause.

Missing values in the track title and artist columns are replaced with the string 'unknown'.

In [9]:
df['track_name'] = df['track_name'].fillna('unknown')

In [10]:
df['artist_name'] = df['artist_name'].fillna('unknown')

In [11]:
df.isnull().sum()

user_id           0
track_name        0
artist_name       0
genre_name     1198
city              0
time              0
weekday           0
dtype: int64

Next, a check for duplicates: any found are removed, and the check is repeated to confirm they're gone.

In [12]:
# get the total number of duplicates in df
df.duplicated().sum()

3826

In [13]:
# remove all duplicates from df
df = df.drop_duplicates().reset_index(drop=True)

In [14]:
df.duplicated().sum()

0

Duplicates may have appeared because of a failure in recording the data. It's worth looking into what causes this kind of "information garbage."

The unique values of the genre column are saved to the variable *genres_list*.

The function *find_genre()* looks for implicit duplicates in the genre column, for example, when the same genre is spelled in different ways.

In [15]:
# save the list of unique values in the 'genre_name' column to genres_list
genres_list = df['genre_name'].unique()

In [16]:
# create the function find_genre()
# the function takes a string with the name of the genre to look for
# in the body, a counter variable is declared and set to 0
# then a for loop goes through the list of unique values
# if the current list item equals the function's parameter,
# the counter goes up by 1
# when the loop ends, the function returns the counter

def find_genre(genre_name):
    genre_count = 0
    for element in genres_list:
        if element == genre_name:
            genre_count += 1
    return genre_count

Calling *find_genre()* to look for different spellings of the hip-hop genre in the table.

The correct name is *hiphop*. Other variants to look for:

* hip
* hop
* hip-hop

In [17]:
# call find_genre() to check for the variant 'hip'
find_genre('hip')

1

In [18]:
# check for the variant 'hop'
find_genre('hop')

0

In [19]:
# check for the variant 'hip-hop'
find_genre('hip-hop')

0

The function *find_hip_hop()* replaces the wrong name of this genre in the *'genre_name'* column with *'hiphop'* and checks that the replacement worked.

This fixes every spelling the check found.

In [20]:
# create the function find_hip_hop()
# the function takes the table df and the wrong name as parameters
# a special method is applied to the 'genre_name' column
# that replaces the second parameter with the string 'hiphop'
# the result is the number of values in the column, counted with count(),
# that equal the second parameter
# the function returns the result

def find_hip_hop(df,wrong):
    df['genre_name'] = df['genre_name'].replace(wrong,'hiphop')
    result = df[df['genre_name'] == wrong]['genre_name'].count()
    return result

In [21]:
# replace one wrong variant with hiphop by calling find_hip_hop()
find_hip_hop(df,'hip')

0

General information about the data confirms that the cleanup worked.

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61253 entries, 0 to 61252
Data columns (total 7 columns):
user_id        61253 non-null object
track_name     61253 non-null object
artist_name    61253 non-null object
genre_name     60126 non-null object
city           61253 non-null object
time           61253 non-null object
weekday        61253 non-null object
dtypes: object(7)
memory usage: 3.3+ MB


During preprocessing, the data turned out to have not only missing values and problems with the column names but also all kinds of duplicates. Removing them makes the analysis more accurate. Since the genre information is important to keep for the analysis, the missing values aren't simply dropped; the missing artist names and track titles are filled in. The column names are now correct and convenient for further work.

# Do people in different cities really listen to music differently?

The hypothesis was that users in Moscow and St. Petersburg listen to music differently. The test uses data for three days of the week: Monday, Wednesday, and Friday.

For each city, the number of tracks with a known genre played on these days is counted and compared.

The data is grouped by city, counting the tracks whose genre is known.

In [23]:
# group df by the 'city' column and count the values in the 'genre_name' column
df.groupby('city')
df.groupby('city').count()

,user_id,track_name,artist_name,genre_name,time,weekday
city,,,,,,
Moscow,42741,42741,42741,41892,42741,42741
Saint-Petersburg,18512,18512,18512,18234,18512,18512


There are more plays in Moscow than in St. Petersburg, but that doesn't mean Moscow is more active. Yandex.Music has more users in Moscow overall, so the numbers are comparable.

The data is grouped by day of the week, counting the tracks with a known genre played on Monday, Wednesday, and Friday.

In [24]:
# group the data by the 'weekday' column and count the values in the 'genre_name' column
df.groupby('weekday')
df.groupby('weekday').count()

,user_id,track_name,artist_name,genre_name,city,time
weekday,,,,,,
Friday,21840,21840,21840,21482,21840,21840
Monday,21354,21354,21354,20866,21354,21354
Wednesday,18059,18059,18059,17778,18059,18059


Monday and Friday are the time for music; on Wednesdays, users are a bit more focused on work.

The function *number_tracks()* takes a table, a day of the week, and a city name, and returns the number of tracks played that have a known genre. It counts the tracks played in each city on Monday, then on Wednesday and Friday.

In [25]:
# create the function number_tracks()
# the function is declared with three parameters: df, day, city
# track_list stores the rows of df where
# the value in the 'weekday' column equals the day parameter
# and, at the same time, the value in the 'city' column equals the city parameter
# track_list_count stores the number of values in the 'genre_name' column,
# calculated with count() for the track_list table
# the function returns track_list_count

def number_tracks(df,day,city):
    track_list = df[(df['weekday'] == day) & (df['city'] == city)]
    track_list_count = track_list['genre_name'].count()
    return track_list_count

In [26]:
# tracks played in Moscow on Monday
number_tracks(df,'Monday','Moscow')

15347

In [27]:
# tracks played in St. Petersburg on Monday
number_tracks(df,'Monday','Saint-Petersburg')

5519

In [28]:
# tracks played in Moscow on Wednesday
number_tracks(df,'Wednesday','Moscow')

10865

In [29]:
# tracks played in St. Petersburg on Wednesday
number_tracks(df,'Wednesday','Saint-Petersburg')

6913

In [30]:
# tracks played in Moscow on Friday
number_tracks(df,'Friday','Moscow')

15680

In [31]:
# tracks played in St. Petersburg on Friday
number_tracks(df,'Friday','Saint-Petersburg')

5802

All of this goes into one table with the column names ['city', 'monday', 'wednesday', 'friday'].

In [32]:
table = pd.DataFrame(data = [['Moscow', 15347, 10865, 15680], ['Saint-Petersburg', 5519, 6913, 5802]],
                     columns = ['city', 'monday', 'wednesday', 'friday'])

The results show that, relative to Wednesday, St. Petersburg and Moscow listen to music as "mirror images": in Moscow, the peaks are on Monday and Friday, and listening drops on Wednesday. In St. Petersburg, Wednesday is the day of greatest interest in music, and interest is lower on Monday and Friday, by almost the same amount.

# Monday morning and Friday evening: different music or the same?

Which genres dominate in each city on Monday morning and on Friday evening? The assumption is that on Monday morning users listen to more upbeat music (for example, pop), and on Friday evening to more dance music (for example, electronic).

Separate tables for Moscow, *moscow_general*, and for St. Petersburg, *spb_general*.

In [33]:
# get the moscow_general table from the rows of df
# where the value in the 'city' column is 'Moscow'

moscow_general = df[df['city'] == 'Moscow']

In [34]:
# get the spb_general table
spb_general = df[df['city'] == 'Saint-Petersburg']

The function *genre_weekday()* returns the list of genres for a given day of the week and time range, from one hour to another.

In [35]:
# declare the function genre_weekday() with parameters df, day, time1, time2
# genre_list stores the rows of df where all of these are true:
# 1) the value in the 'weekday' column equals the day parameter
# 2) the value in the 'time' column is greater than time1, and
# 3) less than time2.
# genre_list_sorted stores, in descending order,
# the first 10 values of the Series obtained by counting 'genre_name' values
# grouped by the 'genre_name' column of genre_list
# the function returns genre_list_sorted

def genre_weekday(df, day, time1, time2):
    genre_list = df[(df['weekday'] == day) & (df['time'] > time1) & (df['time'] < time2)]
    genre_list_sorted = genre_list.groupby('genre_name')['genre_name'].count()
    return genre_list_sorted

Comparing the results for Moscow and St. Petersburg on Monday morning (7 to 11 a.m.) and Friday evening (5 to 11 p.m.).

In [36]:
# call the function for Monday morning in Moscow
genre_weekday(moscow_general, 'Monday','07:00:00','11:00:00')

genre_name
adult            1
africa           2
alternative    164
ambient         22
americana        1
              ... 
videogame        7
vocal            8
western         10
world          181
worldbeat        1
Name: genre_name, Length: 152, dtype: int64

In [37]:
# call the function for Monday morning in St. Petersburg
genre_weekday(spb_general, 'Monday','07:00:00','11:00:00')

genre_name
adult           1
alternative    58
ambient         5
balkan          1
beats           1
               ..
variété         3
videogame       1
vocal           2
western         7
world          36
Name: genre_name, Length: 107, dtype: int64

In [38]:
# call the function for Friday evening in Moscow
genre_weekday(moscow_general, 'Friday','17:00:00','23:00:00')

genre_name
adult            1
africa           1
alternative    163
ambient         18
americana        2
              ... 
variété          6
videogame        5
vocal            9
western          4
world          208
Name: genre_name, Length: 163, dtype: int64

In [39]:
# call the function for Friday evening in St. Petersburg
genre_weekday(spb_general, 'Friday','17:00:00','23:00:00')

genre_name
acoustic        1
adult           1
alternative    63
ambient         7
americana       1
               ..
variété         4
videogame       3
vocal           2
western         5
world          54
Name: genre_name, Length: 126, dtype: int64

The popular genres on Monday morning in St. Petersburg and Moscow turned out to be similar: as expected, pop is popular everywhere. Still, the bottom of the top 10 differs between the two cities: in St. Petersburg, it includes jazz and Russian rap, and in Moscow, the *world* genre.

At the end of the week, the picture doesn't change. Pop is still in first place. Again, the difference shows only at the bottom of the top 10, where St. Petersburg also has the *world* genre on Friday evening.

Pop is the undisputed leader, and the top 5 is basically the same in both capitals. At the same time, the bottom of the list is more "alive": each city has more distinctive genres that really do change position depending on the day of the week and the time.

# Moscow and St. Petersburg: two different capitals, two different directions in music. True?

Hypothesis: St. Petersburg is rich in rap culture, so people listen to rap more often there, while Moscow is a city of contrasts, but most users listen to pop.

The *moscow_general* table is grouped by genre, the tracks of each genre are counted and sorted in descending order, and the result is saved in the table *moscow_genres*.

The first 10 rows of this new table:

In [40]:
moscow_genres = moscow_general.groupby('genre_name')['genre_name'].count().sort_values(ascending = False)
moscow_genres.head(10)

genre_name
pop            5892
dance          4435
rock           3965
electronic     3786
hiphop         2096
classical      1616
world          1432
alternative    1379
ruspop         1372
rusrap         1161
Name: genre_name, dtype: int64

The *spb_general* table is grouped the same way, and the result is saved in the table *spb_genres*.

Its first 10 rows, for comparing the two cities:

In [41]:
spb_genres = spb_general.groupby('genre_name')['genre_name'].count().sort_values(ascending = False)
spb_genres.head(10)

genre_name
pop            2431
dance          1932
rock           1879
electronic     1736
hiphop          960
alternative     649
classical       646
rusrap          564
ruspop          538
world           515
Name: genre_name, dtype: int64

In Moscow, besides pop, which is popular everywhere, there's also the Russian pop genre, which means interest in it is broader there. And rap, contrary to the assumption, holds similar positions in both cities.

# Research results

Working hypotheses:

* people in the two cities, Moscow and St. Petersburg, listen to music on different schedules

* the top-10 genre lists for Monday morning and Friday evening are noticeably different

* people in the two cities prefer different music genres

**Overall results**

Moscow and St. Petersburg share the same tastes: popular music dominates everywhere. At the same time, within each city, preferences don't depend on the day of the week; people always listen to what they like. But between the cities, there's a mirror pattern around Wednesday: Moscow listens more on Monday and Friday, while St. Petersburg is the opposite, listening more on Wednesday and less on Monday and Friday.

As a result, the second hypothesis is confirmed: the top-10 genre lists for Monday morning and Friday evening are noticeably different.